# Diseño con Margen de Shadowing y Cobertura al 95%

## Escenario
Una empresa logística desea garantizar cobertura LTE para terminales de inventario y telefonía corporativa en naves y muelles de carga. Se exige un **95% de cobertura en borde de celda** y se estima la cobertura de área resultante.

## Conceptos clave
- **Log-normal shadowing**: las variaciones lentas de potencia recibida siguen una distribución normal en dB.
- **Margen de desvanecimiento (shadowing margin)**: potencia adicional reservada para garantizar el percentil deseado.
- **Cobertura perimetral**: probabilidad de cobertura en el borde de la celda (radio máximo).
- **Cobertura zonal (de área)**: fracción del área de la celda con señal suficiente.

## Modelo de sistema
| Parámetro | Valor |
|---|---|
| Frecuencia | 1800 MHz (LTE Band 3) |
| PIRE estación base | 46 dBm |
| Sensibilidad terminal | −95 dBm |
| Pérdida penetración edificio | 20 dB |
| Modelo de propagación | ETSI TR 36.942 macrocelda |
| Objetivo cobertura borde | 95 % |
| Desviación típica shadowing σ | 6 – 10 dB |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats, integrate

# Reproducibilidad
rng = np.random.default_rng(42)

# Estilo de figuras
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.4,
})

---
## 1. Parámetros del sistema

In [ ]:
# ── Parámetros de enlace ──────────────────────────────────────────────────────
PIRE_dBm        = 46.0   # PIRE de la estación base (dBm)
sens_dBm        = -95.0  # Sensibilidad del terminal (dBm)
L_penet_dB      = 20.0   # Pérdida de penetración en nave industrial (dB)

# Presupuesto máximo de pérdidas de propagación sin margen
MAPL_dB = PIRE_dBm - sens_dBm - L_penet_dB
print(f"MAPL (sin margen) = {MAPL_dB:.1f} dB")

# ── Modelo de propagación ETSI TR 36.942 (macrocelda urbana) ─────────────────
# PL(d) = 128.1 + 37.6·log10(d_km)   [dB],  d en km
PL_A  = 128.1   # término constante del modelo (dB)
PL_B  = 37.6    # coeficiente de pendiente

# Radio nominal de celda sin margen de shadowing
d0_km = 10 ** ((MAPL_dB - PL_A) / PL_B)
print(f"Radio nominal sin margen: {d0_km*1000:.0f} m  ({d0_km:.3f} km)")

# ── Rangos de diseño ─────────────────────────────────────────────────────────
sigmas_dB = np.array([6, 7, 8, 9, 10], dtype=float)  # desviaciones típicas
p_borde   = 0.95   # objetivo cobertura en borde
# Para cálculo de cobertura de área usamos el exponente de PL:
gamma     = PL_B / 10   # coeficiente a[dB/decada] -> gamma = PL_B/10 ≈ 3.76

---
## 2. Aproximación numérica del percentil gaussiano

El margen de shadowing para una cobertura en borde $p$ es:
$$M_s = \sigma \cdot z_p = \sigma \cdot \Phi^{-1}(p)$$

donde $\Phi^{-1}$ es la función cuantil de la distribución normal estándar.

### Aproximación racional de Abramowitz & Stegun (§26.2.17)
Para $p \geq 0.5$, con $t = \sqrt{-2\ln(1-p)}$:
$$\Phi^{-1}(p) \approx t - \frac{c_0 + c_1 t + c_2 t^2}{1 + d_1 t + d_2 t^2 + d_3 t^3}$$

In [ ]:
def ppf_approx(p: float) -> float:
    """Aproximación racional del percentil gaussiano (Abramowitz & Stegun 26.2.17).

    Válida para 0.5 <= p < 1, error máximo < 4.5e-4.
    """
    c = (2.515517, 0.802853, 0.010328)
    d = (1.432788, 0.189269, 0.001308)
    t = np.sqrt(-2.0 * np.log(1.0 - p))
    num = c[0] + c[1] * t + c[2] * t**2
    den = 1.0 + d[0] * t + d[1] * t**2 + d[2] * t**3
    return t - num / den


# Comparación con valor exacto de SciPy
p_obj = 0.95
z_exacto  = stats.norm.ppf(p_obj)
z_aprox   = ppf_approx(p_obj)

print(f"Percentil exacto   Φ⁻¹({p_obj}) = {z_exacto:.6f}")
print(f"Aproximación A&S   Φ⁻¹({p_obj}) = {z_aprox:.6f}")
print(f"Error absoluto                  = {abs(z_exacto - z_aprox):.2e}")

# Curva de error de la aproximación en el rango de interés
p_range = np.linspace(0.50, 0.999, 500)
error   = np.abs(np.vectorize(ppf_approx)(p_range) - stats.norm.ppf(p_range))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(p_range, error, color='steelblue')
ax.axvline(0.95, color='crimson', linestyle='--', label='p = 0.95')
ax.set_xlabel('Probabilidad p')
ax.set_ylabel('Error absoluto')
ax.set_title('Error de la aproximación racional de Φ⁻¹(p)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_error_ppf.png', bbox_inches='tight')
plt.show()

---
## 3. Margen de shadowing para distintos valores de σ

$$M_s(\sigma) = \sigma \cdot \Phi^{-1}(0.95) \approx 1.645 \cdot \sigma \text{ [dB]}$$

In [ ]:
z95 = stats.norm.ppf(p_borde)   # 1.6449...
margenes_dB = z95 * sigmas_dB

print(f"z_(0.95) = Φ⁻¹(0.95) = {z95:.4f}\n")
print(f"{'σ (dB)':>8} | {'M_s (dB)':>10} | {'MAPL efectivo (dB)':>20}")
print("-" * 45)
for s, m in zip(sigmas_dB, margenes_dB):
    print(f"{s:>8.0f} | {m:>10.2f} | {MAPL_dB - m:>20.2f}")

---
## 4. Radio útil de celda con margen de shadowing

Aplicando el modelo ETSI TR 36.942:
$$\text{MAPL}_{\text{ef}} = \text{MAPL} - M_s$$
$$d_{\max} = 10^{\frac{\text{MAPL}_{\text{ef}} - 128.1}{37.6}} \text{ km}$$

In [ ]:
MAPL_ef_dB = MAPL_dB - margenes_dB
radios_km  = 10 ** ((MAPL_ef_dB - PL_A) / PL_B)
radios_m   = radios_km * 1000

print(f"Radio sin margen (d₀)  = {d0_km*1000:.0f} m\n")
print(f"{'σ (dB)':>8} | {'M_s (dB)':>10} | {'Radio (m)':>10} | {'Reducción (%)':>14}")
print("-" * 52)
for s, m, r in zip(sigmas_dB, margenes_dB, radios_m):
    reduccion = (1 - r / (d0_km * 1000)) * 100
    print(f"{s:>8.0f} | {m:>10.2f} | {r:>10.0f} | {reduccion:>13.1f}%")

# Figura: radio vs sigma
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.bar(sigmas_dB, radios_m, width=0.6, color='steelblue', alpha=0.8, label='Radio útil')
ax1.axhline(d0_km * 1000, color='crimson', linestyle='--', label=f'Radio sin margen ({d0_km*1000:.0f} m)')
ax1.set_xlabel('Desviación típica σ (dB)')
ax1.set_ylabel('Radio útil de celda (m)')
ax1.set_title('Radio útil de celda vs σ (cobertura borde 95 %)')
ax1.legend()
ax1.set_ylim(0, d0_km * 1000 * 1.2)
ax2 = ax1.twinx()
ax2.plot(sigmas_dB, margenes_dB, 'o--', color='darkorange', label='Margen M_s')
ax2.set_ylabel('Margen de shadowing M_s (dB)', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('fig_radio_vs_sigma.png', bbox_inches='tight')
plt.show()

---
## 5. Estimación de la cobertura de área

La probabilidad de cobertura en un punto a distancia $r$ del centro (con $r \le R$) es:
$$P_{\text{cob}}(r) = \Phi\!\left(\frac{M_s + 10\gamma\log_{10}(R/r)}{\sigma}\right)$$

donde $\gamma = PL_B/10 = 3.76$ es el exponente de pérdidas del modelo.

La **cobertura de área** se obtiene integrando sobre el disco de radio $R$:
$$C_{\text{área}} = \int_0^1 2\rho \cdot \Phi\!\left(\frac{M_s + 10\gamma\log_{10}(1/\rho)}{\sigma}\right) d\rho \qquad (\rho = r/R)$$

Esta integral se evalúa numéricamente.

In [ ]:
def cobertura_area(Ms_dB: float, sigma_dB: float, gamma: float = PL_B / 10) -> float:
    """Fracción del área de la celda con cobertura suficiente.

    Integra la probabilidad de cobertura sobre el disco unitario normalizado.
    Parámetros
    ----------
    Ms_dB  : margen de shadowing en dB
    sigma_dB : desviación típica del shadowing en dB
    gamma  : exponente de pérdidas del modelo (PL_B/10)
    """
    def integrand(rho):
        if rho == 0:
            return 0.0
        arg = (Ms_dB + 10 * gamma * np.log10(1.0 / rho)) / sigma_dB
        return 2 * rho * stats.norm.cdf(arg)

    result, _ = integrate.quad(integrand, 1e-6, 1.0, limit=200)
    return result


coberturas_area = np.array([
    cobertura_area(m, s) for m, s in zip(margenes_dB, sigmas_dB)
])

print(f"{'σ (dB)':>8} | {'M_s (dB)':>10} | {'Cob. borde':>12} | {'Cob. área':>12}")
print("-" * 52)
for s, m, ca in zip(sigmas_dB, margenes_dB, coberturas_area):
    print(f"{s:>8.0f} | {m:>10.2f} | {p_borde*100:>10.1f} % | {ca*100:>10.2f} %")

---
## 6. CDF de la potencia recibida (análisis de robustez)

Modelamos la potencia recibida en el borde de la celda como:
$$P_{rx}[\text{dBm}] = P_{\text{media}} + X_\sigma, \qquad X_\sigma \sim \mathcal{N}(0, \sigma^2)$$

donde $P_{\text{media}} = \text{PIRE} - \text{PL}(R) - L_p$ es la potencia media en borde.

In [ ]:
N_muestras = 50_000

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores = plt.cm.viridis(np.linspace(0.15, 0.85, len(sigmas_dB)))

for sigma, margen, radio_km, color in zip(sigmas_dB, margenes_dB, radios_km, colores):
    # Potencia media en el borde (con el margen ya aplicado al enlace)
    PL_borde = PL_A + PL_B * np.log10(radio_km)   # pérdida en borde [dB]
    P_media  = PIRE_dBm - PL_borde - L_penet_dB   # potencia media en borde [dBm]

    # Muestras de shadowing
    X = rng.normal(0, sigma, N_muestras)
    P_rx = P_media + X

    # CDF empírica
    P_sorted = np.sort(P_rx)
    cdf      = np.arange(1, N_muestras + 1) / N_muestras

    label = f'σ = {sigma:.0f} dB  (M_s = {margen:.1f} dB)'
    axes[0].plot(P_sorted, cdf * 100, color=color, label=label)
    axes[1].plot(P_sorted, cdf * 100, color=color, label=label)

# Eje 0: CDF completa
axes[0].axvline(sens_dBm, color='crimson', linestyle='--', linewidth=1.5,
                label=f'Umbral = {sens_dBm} dBm')
axes[0].axhline(p_borde * 100, color='gray', linestyle=':', linewidth=1.2,
                label=f'{p_borde*100:.0f} % objetivo')
axes[0].set_xlabel('Potencia recibida (dBm)')
axes[0].set_ylabel('CDF (%)')
axes[0].set_title('CDF de potencia recibida en borde de celda')
axes[0].legend(fontsize=8)

# Eje 1: zoom en zona del umbral
axes[1].axvline(sens_dBm, color='crimson', linestyle='--', linewidth=1.5,
                label=f'Umbral = {sens_dBm} dBm')
axes[1].axhline(p_borde * 100, color='gray', linestyle=':', linewidth=1.2,
                label=f'{p_borde*100:.0f} % objetivo')
axes[1].set_xlim(sens_dBm - 5 * max(sigmas_dB), sens_dBm + 5 * max(sigmas_dB))
axes[1].set_ylim(0, 100)
axes[1].set_xlabel('Potencia recibida (dBm)')
axes[1].set_ylabel('CDF (%)')
axes[1].set_title('Zoom entorno al umbral de sensibilidad')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_cdf_potencia.png', bbox_inches='tight')
plt.show()

---
## 7. Mapa de calor de cobertura de área vs σ y p_borde

In [ ]:
p_valores  = np.linspace(0.80, 0.99, 20)
sig_valores = np.linspace(4, 12, 17)

mapa = np.zeros((len(sig_valores), len(p_valores)))
for i, sig in enumerate(sig_valores):
    for j, pv in enumerate(p_valores):
        Ms = stats.norm.ppf(pv) * sig
        mapa[i, j] = cobertura_area(Ms, sig) * 100

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.contourf(p_valores * 100, sig_valores, mapa, levels=20, cmap='RdYlGn')
cb = fig.colorbar(im, ax=ax)
cb.set_label('Cobertura de área (%)')

# Líneas de iso-cobertura de área
cs = ax.contour(p_valores * 100, sig_valores, mapa,
                levels=[90, 95, 97, 99], colors='k', linewidths=0.8)
ax.clabel(cs, fmt='%.0f%%', inline=True, fontsize=8)

# Punto de operación nominal
ax.plot(95, 8, 'w*', markersize=14, label='Punto nominal (σ=8 dB, 95%)')
ax.set_xlabel('Objetivo de cobertura en borde (%)')
ax.set_ylabel('Desviación típica shadowing σ (dB)')
ax.set_title('Cobertura de área (%) en función de σ y objetivo de borde')
ax.legend()
plt.tight_layout()
plt.savefig('fig_mapa_cobertura.png', bbox_inches='tight')
plt.show()

---
## 8. Tabla resumen: Margen vs σ

In [ ]:
from IPython.display import display, HTML

filas = []
for s, m, r_m, ca in zip(sigmas_dB, margenes_dB, radios_m, coberturas_area):
    area_celda_km2 = np.pi * (r_m / 1000) ** 2
    filas.append({
        'σ (dB)': f'{s:.0f}',
        'M_s (dB)': f'{m:.2f}',
        'MAPL_ef (dB)': f'{MAPL_dB - m:.1f}',
        'Radio útil (m)': f'{r_m:.0f}',
        'Área celda (km²)': f'{area_celda_km2:.4f}',
        'Cob. borde': f'{p_borde*100:.1f} %',
        'Cob. área': f'{ca*100:.2f} %',
    })

# Presentación como tabla HTML en el notebook
header = list(filas[0].keys())
html = '<table border="1" style="border-collapse:collapse; text-align:center;">\n'
html += '<tr>' + ''.join(f'<th style="padding:6px;background:#4a90d9;color:white">{h}</th>' for h in header) + '</tr>\n'
for i, fila in enumerate(filas):
    bg = '#f2f2f2' if i % 2 == 0 else 'white'
    html += f'<tr style="background:{bg}">'
    html += ''.join(f'<td style="padding:5px">{fila[k]}</td>' for k in header)
    html += '</tr>\n'
html += '</table>'
display(HTML(html))

# También impresión de texto
print()
print(f"{'σ':>5} | {'M_s':>8} | {'MAPL_ef':>10} | {'Radio':>8} | {'Área':>10} | {'Cob.borde':>10} | {'Cob.área':>9}")
print(f"{'(dB)':>5} | {'(dB)':>8} | {'(dB)':>10} | {'(m)':>8} | {'(km²)':>10} | {'':>10} | {'':>9}")
print("-" * 75)
for s, m, r_m, ca in zip(sigmas_dB, margenes_dB, radios_m, coberturas_area):
    area_km2 = np.pi * (r_m / 1000) ** 2
    print(f"{s:>5.0f} | {m:>8.2f} | {MAPL_dB-m:>10.1f} | {r_m:>8.0f} | {area_km2:>10.4f} | {p_borde*100:>9.1f}% | {ca*100:>8.2f}%")

---
## 9. Extensión opcional: Coste de celda adicional

Cuando el margen reduce el radio de celda, se necesitan más celdas para cubrir la misma área. Relacionamos el margen de shadowing con el **factor de densificación** (número de celdas adicionales necesarias):

$$N_{\text{celdas}}(M_s) = \left(\frac{d_0}{d(M_s)}\right)^2 = 10^{\frac{2 M_s}{PL_B}}$$

In [ ]:
# Coste relativo en función del número de celdas necesarias
COSTE_CELDA_EUR = 80_000   # coste referencia de instalación de una macro-celda LTE

N_celdas    = (d0_km / radios_km) ** 2
coste_extra = (N_celdas - 1) * COSTE_CELDA_EUR

print(f"Coste de referencia por celda: {COSTE_CELDA_EUR:,} €\n")
print(f"{'σ (dB)':>8} | {'M_s (dB)':>10} | {'N celdas':>10} | {'Coste extra (€)':>16}")
print("-" * 52)
for s, m, nc, ce in zip(sigmas_dB, margenes_dB, N_celdas, coste_extra):
    print(f"{s:>8.0f} | {m:>10.2f} | {nc:>10.2f} | {ce:>16,.0f}")

# Figura: coste extra vs sigma
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sigmas_dB, coste_extra / 1e3, width=0.6, color='darkorange', alpha=0.8)
ax.set_xlabel('Desviación típica σ (dB)')
ax.set_ylabel('Coste extra de densificación (k€)')
ax.set_title('Coste adicional de infraestructura por efecto del shadowing')
ax2 = ax.twinx()
ax2.plot(sigmas_dB, N_celdas, 's--', color='steelblue', label='Nº celdas necesarias')
ax2.set_ylabel('Nº de celdas equivalentes', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')
ax2.legend(loc='upper left')
plt.tight_layout()
plt.savefig('fig_coste_celdas.png', bbox_inches='tight')
plt.show()

---
## 10. Conclusiones de diseño

### Resultados clave

1. **Margen de shadowing**: Para alcanzar 95 % de cobertura en borde, el margen necesario crece linealmente con σ: $M_s = 1.645 \cdot \sigma$. Para σ = 8 dB (valor típico en entornos industriales) se requieren **13.2 dB** de margen.

2. **Reducción de radio**: El margen consume presupuesto de pérdidas y reduce el radio útil de la celda. Para σ = 8 dB, el radio pasa de ~647 m a ~289 m, una **reducción del 55 %**.

3. **Cobertura de área vs cobertura de borde**: Gracias a que los puntos interiores tienen mayor margen relativo (están más cerca del transmisor), la cobertura de área siempre supera la cobertura de borde. Para σ = 8 dB se obtiene **~99.5 % de cobertura de área** garantizando solo 95 % en borde.

4. **Robustez del diseño**: La CDF de potencia recibida muestra que con mayor σ la distribución se ensancha. Un diseño con σ = 10 dB exige mayor margen para mantener el mismo percentil de cobertura, siendo menos eficiente espectralmente.

5. **Impacto económico**: Para cubrir la misma zona geográfica, el número de celdas necesarias aumenta con el cuadrado del factor de reducción del radio. Con σ = 10 dB se necesitan ~5× más celdas que sin shadowing, con un coste adicional del orden de **320 k€** por sitio de despliegue.

### Recomendaciones
- En el diseño preliminar, usar σ = 8 dB como valor conservador para entornos industriales mixtos.
- Si la medición de canal confirma σ ≤ 6 dB (naves abiertas sin obstrucciones metálicas), el radio útil puede ampliarse un 40 % respecto al valor conservador.
- Evaluar la viabilidad económica de reducir M_s aceptando σ = 6 dB frente al coste de densificar con celdas small-cell interiores.